# Real-World Pipeline: Document Processing System

A complex pipeline that combines every misen operator:

```
Input documents
  → classify (branch by type)
  → process each type differently
  → extract metadata in parallel
  → quality check loop
  → final output
```

No external dependencies — everything runs with simulated data.

## Setup

In [ ]:
!cd .. && uv pip install -e ".[dev]" --quiet

In [ ]:
from misen import (
    tool,
    sequential,
    parallel,
    branch,
    loop,
    map_each,
)

print("misen imported OK")

## Step 1: Define atomic blocks

Each block does one thing. They don't know about each other.

In [ ]:
import hashlib
import re
from datetime import datetime


# ── Document loading & classification ──

@tool(name="load_document")
def load_document(input: dict) -> dict:
    """Load a document and detect its type."""
    path = input["file_path"]
    ext = path.rsplit(".", 1)[-1].lower()

    # Simulate loading different file types
    content_db = {
        "quarterly_report.pdf": "Q3 Revenue: $4.2M (+23% YoY). EBITDA: $1.18M. Headcount: 48 FTEs. CAC: $38 (down from $45). MAU: 18,500.",
        "customer_feedback.txt": "User A: Great product, love the UI! User B: Loading is slow. User C: Support was helpful. User D: Missing dark mode. User E: Best tool I've used this year.",
        "api_spec.json": '{"openapi": "3.0", "paths": {"/users": {"get": {"summary": "List users"}}, "/orders": {"post": {"summary": "Create order"}}}}',
        "meeting_notes.md": "# Sprint Planning\n- Migrate auth to OAuth2\n- Fix search indexing bug\n- Design new dashboard\n## Action items\n- @alice: OAuth2 RFC by Friday\n- @bob: search hotfix today\n- @carol: dashboard mockups",
        "sales_data.csv": "month,revenue,deals\nJan,320000,12\nFeb,350000,15\nMar,410000,18\nApr,380000,14\nMay,420000,19\nJun,450000,21",
    }

    text = content_db.get(path, f"Content of {path}")

    type_map = {"pdf": "report", "txt": "feedback", "json": "structured", "md": "notes", "csv": "data"}
    doc_type = type_map.get(ext, "unknown")

    return {
        "text": text,
        "doc_type": doc_type,
        "file_ext": ext,
        "char_count": len(text),
    }


# ── Text processing ──

@tool(name="chunk_text")
def chunk_text(input: dict) -> dict:
    """Split text into semantic chunks."""
    text = input["text"]
    # Split on sentence boundaries or newlines
    sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    # Group into chunks of ~3 sentences
    chunks = []
    for i in range(0, len(sentences), 3):
        chunk = " ".join(sentences[i:i+3])
        chunks.append(chunk)

    return {"chunks": chunks, "chunk_count": len(chunks)}


@tool(name="generate_embeddings")
def generate_embeddings(input: dict) -> dict:
    """Generate embedding vectors for chunks."""
    chunks = input["chunks"]
    # Simulate embeddings (deterministic hash-based)
    vectors = []
    for chunk in chunks:
        h = hashlib.md5(chunk.encode()).hexdigest()  # noqa: S324
        vec = [int(h[i:i+2], 16) / 255.0 for i in range(0, 16, 2)]
        vectors.append(vec)

    return {"vectors": vectors, "embedding_dim": len(vectors[0]) if vectors else 0}

In [ ]:
# ── Metadata extraction (these run in parallel) ──

@tool(name="extract_keywords")
def extract_keywords(input: dict) -> dict:
    """Extract keywords from text."""
    text = input["text"].lower()
    # Simple keyword extraction: words that appear important
    stop_words = {"the", "a", "an", "is", "are", "was", "were", "to", "of", "in", "for", "and", "or", "by"}
    words = re.findall(r'\b[a-z]+\b', text)
    freq = {}
    for w in words:
        if w not in stop_words and len(w) > 3:
            freq[w] = freq.get(w, 0) + 1
    keywords = sorted(freq, key=freq.get, reverse=True)[:10]
    return {"keywords": keywords}


@tool(name="detect_language")
def detect_language(input: dict) -> dict:
    """Detect the language of the text."""
    text = input["text"]
    # Simple heuristic
    korean_chars = len(re.findall(r'[가-힣]', text))
    if korean_chars > len(text) * 0.3:
        return {"language": "ko"}
    return {"language": "en"}


@tool(name="compute_stats")
def compute_stats(input: dict) -> dict:
    """Compute basic statistics about the document."""
    text = input["text"]
    words = text.split()
    sentences = re.split(r'[.!?]+', text)
    return {
        "stats": {
            "word_count": len(words),
            "sentence_count": len([s for s in sentences if s.strip()]),
            "avg_word_length": round(sum(len(w) for w in words) / max(len(words), 1), 1),
            "has_numbers": bool(re.search(r'\d', text)),
        }
    }


@tool(name="generate_doc_id")
def generate_doc_id(input: dict) -> dict:
    """Generate a unique document ID."""
    text = input["text"]
    doc_hash = hashlib.sha256(text.encode()).hexdigest()[:12]
    return {"doc_id": f"doc_{doc_hash}"}

In [ ]:
# ── Type-specific processors ──

@tool(name="process_report")
def process_report(input: dict) -> dict:
    """Extract structured data from reports."""
    text = input["text"]
    # Extract numbers and metrics
    numbers = re.findall(r'\$[\d,.]+[MBK]?|[\d.]+%|[\d,]+', text)
    return {
        "extracted_metrics": numbers[:10],
        "processing_note": "Report: extracted financial metrics",
    }


@tool(name="process_feedback")
def process_feedback(input: dict) -> dict:
    """Analyze customer feedback."""
    text = input["text"]
    positive = len(re.findall(r'\b(great|love|best|helpful|excellent)\b', text, re.I))
    negative = len(re.findall(r'\b(slow|missing|bad|terrible|worst)\b', text, re.I))
    total = positive + negative or 1
    return {
        "sentiment_score": round(positive / total, 2),
        "positive_mentions": positive,
        "negative_mentions": negative,
        "processing_note": "Feedback: sentiment analyzed",
    }


@tool(name="process_structured")
def process_structured(input: dict) -> dict:
    """Parse structured data (JSON, CSV)."""
    import json as json_mod
    text = input["text"]
    try:
        data = json_mod.loads(text)
        return {
            "parsed_structure": list(data.keys()) if isinstance(data, dict) else type(data).__name__,
            "processing_note": "Structured: JSON parsed",
        }
    except json_mod.JSONDecodeError:
        lines = text.strip().split("\n")
        return {
            "parsed_structure": {"rows": len(lines), "columns": len(lines[0].split(",")) if lines else 0},
            "processing_note": "Structured: CSV parsed",
        }


@tool(name="process_notes")
def process_notes(input: dict) -> dict:
    """Extract action items from notes."""
    text = input["text"]
    actions = re.findall(r'- @(\w+): (.+)', text)
    return {
        "action_items": [{"assignee": a, "task": t} for a, t in actions],
        "processing_note": f"Notes: {len(actions)} action items extracted",
    }

In [ ]:
# ── Quality check ──

@tool(name="quality_check")
def quality_check(input: dict) -> dict:
    """Check processing quality and increment score."""
    score = input.get("quality_score", 0)
    checks_passed = 0
    total_checks = 4

    if input.get("keywords"):
        checks_passed += 1
    if input.get("language"):
        checks_passed += 1
    if input.get("doc_id"):
        checks_passed += 1
    if input.get("processing_note"):
        checks_passed += 1

    new_score = checks_passed / total_checks
    return {
        "quality_score": new_score,
        "quality_passed": new_score >= 0.75,
        "checks_passed": checks_passed,
        "total_checks": total_checks,
    }

## Step 2: Compose the pipeline

Now combine all blocks using misen operators.

In [ ]:
# ── Type-specific routing ──
# branch picks the right processor based on doc_type

type_router = branch(
    lambda d: d["doc_type"] == "report",
    process_report,
    branch(
        lambda d: d["doc_type"] == "feedback",
        process_feedback,
        branch(
            lambda d: d["doc_type"] in ("structured", "data"),
            process_structured,
            process_notes,  # default: treat as notes
        ),
    ),
)

# ── Metadata extraction (all run in parallel) ──

metadata_extractor = parallel(
    extract_keywords,
    detect_language,
    compute_stats,
    generate_doc_id,
)

# ── Full pipeline ──

document_pipeline = sequential(
    load_document,          # 1. Load and detect type
    chunk_text,             # 2. Split into chunks
    generate_embeddings,    # 3. Generate embeddings
    type_router,            # 4. Type-specific processing (branch)
    metadata_extractor,     # 5. Extract metadata (parallel)
    quality_check,          # 6. Quality check
)

print(f"Pipeline: {document_pipeline.name}")

## Step 3: Run on different document types

In [ ]:
def print_result(result: dict, label: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Type:       {result['doc_type']}")
    print(f"  Language:   {result['language']}")
    print(f"  Doc ID:     {result['doc_id']}")
    print(f"  Chunks:     {result['chunk_count']}")
    print(f"  Embeddings: {len(result['vectors'])} × {result['embedding_dim']}d")
    print(f"  Keywords:   {', '.join(result['keywords'][:5])}")
    print(f"  Stats:      {result['stats']}")
    print(f"  Processing: {result['processing_note']}")
    print(f"  Quality:    {result['quality_score']:.0%} ({result['checks_passed']}/{result['total_checks']} checks)")
    if "extracted_metrics" in result:
        print(f"  Metrics:    {result['extracted_metrics']}")
    if "sentiment_score" in result:
        print(f"  Sentiment:  {result['sentiment_score']} (+{result['positive_mentions']} -{result['negative_mentions']})")
    if "action_items" in result:
        print(f"  Actions:    {result['action_items']}")
    if "parsed_structure" in result:
        print(f"  Structure:  {result['parsed_structure']}")

In [ ]:
# Process different document types through the same pipeline

files = [
    "quarterly_report.pdf",
    "customer_feedback.txt",
    "api_spec.json",
    "meeting_notes.md",
    "sales_data.csv",
]

for f in files:
    result = document_pipeline.run_sync({"file_path": f})
    print_result(result, f)

## Step 4: Batch processing with `map_each`

Process multiple documents concurrently.

In [ ]:
# Wrap the pipeline to accept an item from map_each
@tool(name="process_one")
async def process_one(input: dict) -> dict:
    """Process a single document from a batch."""
    result = await document_pipeline.run({"file_path": input["item"]})
    return {
        "file": input["item"],
        "doc_type": result["doc_type"],
        "quality": result["quality_score"],
        "chunks": result["chunk_count"],
        "keywords": result["keywords"][:3],
    }


batch_pipeline = map_each(process_one, over_key="files", output_key="results")

batch_result = await batch_pipeline.run({"files": files})

print(f"\nBatch processed {len(batch_result['results'])} documents:\n")
for r in batch_result["results"]:
    print(f"  {r['file']:30s} type={r['doc_type']:12s} quality={r['quality']:.0%}  chunks={r['chunks']}  keywords={r['keywords']}")

## Step 5: Pipeline with quality loop

Retry processing until quality threshold is met.
This simulates an iterative refinement loop.

In [ ]:
@tool(name="enrich")
def enrich(input: dict) -> dict:
    """Enrich document with additional processing each iteration."""
    iteration = input.get("iteration", 0) + 1
    enrichments = input.get("enrichments", [])

    # Each iteration adds more metadata
    new_enrichments = {
        1: "basic_parse",
        2: "entity_extraction",
        3: "cross_reference",
        4: "validation",
    }

    step = new_enrichments.get(iteration, f"extra_{iteration}")
    enrichments = [*enrichments, step]

    return {
        "iteration": iteration,
        "enrichments": enrichments,
        "completeness": min(iteration / 3, 1.0),  # reaches 1.0 at iteration 3
    }


enrichment_loop = loop(
    enrich,
    until=lambda d: d.get("completeness", 0) >= 1.0,
    max_iterations=10,
)

# Combine: load → basic pipeline → iterative enrichment
full_system = sequential(
    load_document,
    metadata_extractor,
    enrichment_loop,
)

result = full_system.run_sync({"file_path": "quarterly_report.pdf"})

print(f"Iterations: {result['iteration']}")
print(f"Completeness: {result['completeness']:.0%}")
print(f"Enrichments: {result['enrichments']}")
print(f"Keywords: {result['keywords'][:5]}")
print(f"Language: {result['language']}")

## Step 6: Reuse sub-pipelines

Every pipeline is a block. Compose them into larger systems.

In [ ]:
@tool(name="compare_documents")
def compare_documents(input: dict) -> dict:
    """Compare two processed document results."""
    results = input["results"]
    if len(results) < 2:
        return {"comparison": "Need at least 2 documents"}

    a, b = results[0], results[1]
    shared_keywords = set(a.get("keywords", [])) & set(b.get("keywords", []))

    return {
        "comparison": {
            "doc_a": a["file"],
            "doc_b": b["file"],
            "same_type": a["doc_type"] == b["doc_type"],
            "shared_keywords": list(shared_keywords),
            "quality_diff": abs(a["quality"] - b["quality"]),
        }
    }


# Compose: batch process → compare
analysis_system = sequential(batch_pipeline, compare_documents)

result = await analysis_system.run({
    "files": ["quarterly_report.pdf", "sales_data.csv"]
})

print("Comparison:")
for k, v in result["comparison"].items():
    print(f"  {k}: {v}")

## Architecture Summary

```
document_pipeline = sequential(
    load_document,                          # tool
    chunk_text,                             # tool
    generate_embeddings,                    # tool
    branch(                                 # branch (nested 3 levels)
        is_report → process_report,
        is_feedback → process_feedback,
        is_structured → process_structured,
        else → process_notes,
    ),
    parallel(                               # parallel (4 concurrent)
        extract_keywords,
        detect_language,
        compute_stats,
        generate_doc_id,
    ),
    quality_check,                          # tool
)

batch_pipeline = map_each(process_one)      # map_each (concurrent)

enrichment_loop = loop(enrich, until=done)  # loop (iterative)

analysis_system = sequential(               # composition
    batch_pipeline,
    compare_documents,
)
```

Every operator used. Every result is a block. Everything composes.